# Comparison of Generative Data Sampling

Code is from [This Github Repo](https://github.com/Diyago/Tabular-data-generation/tree/master) to save time

basically will be running all four options and comparing outputs to show how these vary and if there is noticable difference for the quality of the data and other implications for synthetic data to improve dataset size (like if that causes bias?)


In [1]:
import kagglehub
import pandas as pd
import ipaddress
from sklearn.preprocessing import LabelEncoder

# Download latest version to compare with the two generated datasets from models
path = kagglehub.dataset_download(
    "munaalhawawreh/xiiotid-iiot-intrusion-dataset")

print("Path to dataset files:", path)
print(path)
df = pd.read_csv(path + "\\X-IIoTID dataset.csv", skipinitialspace=True)


# 1. Basic Cleaning: Strip whitespace from columns and string values
df.columns = df.columns.str.strip()
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# 2. Universal IP & Port Normalizer (Handles IPv4, IPv6, ?, -, and Ports)


def universal_to_numeric(val):
    val = str(val).strip()
    if val in ['?', '-', '', 'None']:
        return 0
    try:
        # If it looks like an IP, convert to int
        if '.' in val or ':' in val:
            return int(ipaddress.ip_address(val))
        # Otherwise, treat as a standard number
        return int(float(val))
    except:
        return 0


for col in ['Scr_IP', 'Des_IP', 'Scr_port', 'Des_port']:
    df[col] = df[col].apply(universal_to_numeric)

# 3. Timestamp Fix (Convert Unix to Numeric safely)
df['Timestamp'] = pd.to_numeric(
    df['Timestamp'], errors='coerce').fillna(0).astype(int)
df['Time'] = pd.to_datetime(df['Timestamp'], unit='s')
df.drop(columns=['Timestamp'], inplace=True)
# 4. Boolean/Flag Mapping (TRUE/FALSE strings or actual Bools -> 0/1)
bool_cols = [
    'is_syn_only', 'Is_SYN_ACK', 'is_pure_ack', 'is_with_payload',
    'FIN or RST', 'Bad_checksum', 'is_SYN_with_RST', 'anomaly_alert'
]
for col in bool_cols:
    if col in df.columns:
        # Map strings if they exist, otherwise force to int
        if df[col].dtype == 'object':
            df[col] = df[col].str.upper().map(
                {'TRUE': 1, 'FALSE': 0, '-': 0, '?': 0})
        df[col] = df[col].fillna(0).astype(int)

# 5. Mass Numeric Conversion for Metrics (Duration, Bytes, CPU stats, etc.)
# We replace '?' and '-' with 0 first
df.replace(['?', '-'], 0, inplace=True)
cat_cols = ['Protocol', 'Service', 'class1', 'class2', 'class3']
encoders = {}
# 6. Categorical Cleanup
for col in ['Protocol', 'Service']:
    df[col] = df[col].replace(0, 'unknown').astype(str)
for col in cat_cols:
    if col in df.columns:
        le = LabelEncoder()
        # Ensure everything is a string before encoding to avoid mixed-type errors
        df[col] = le.fit_transform(df[col].astype(str))
        encoders[col] = le

# 3. Final safety check: ensure NO strings remain in the numeric features
# (This prevents the 'udp' error once and for all)
for col in df.columns:
    if df[col].dtype == 'object':
        # If any stray strings exist, force them to numeric or drop
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
# Identify all columns that should be numeric (excluding categorical labels)

numeric_cols = [c for c in df.columns if c not in cat_cols + ['Date']]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# 7. Final Drop
df.drop(columns=['Date'], inplace=True)

df.info()

c:\Users\antho\.conda\envs\CS9840\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\antho\.cache\kagglehub\datasets\munaalhawawreh\xiiotid-iiot-intrusion-dataset\versions\1
C:\Users\antho\.cache\kagglehub\datasets\munaalhawawreh\xiiotid-iiot-intrusion-dataset\versions\1


C:\Users\antho\AppData\Local\Temp\ipykernel_21132\1249797039.py:12: DtypeWarning: Columns (0: Timestamp, 1: Scr_port, 2: Des_port, 3: missed_bytes, 4: anomaly_alert) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path + "\\X-IIoTID dataset.csv", skipinitialspace=True)


<class 'pandas.DataFrame'>
RangeIndex: 820834 entries, 0 to 820833
Data columns (total 67 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Scr_IP                       820834 non-null  float64
 1   Scr_port                     820834 non-null  int64  
 2   Des_IP                       820834 non-null  float64
 3   Des_port                     820834 non-null  int64  
 4   Protocol                     820834 non-null  int64  
 5   Service                      820834 non-null  int64  
 6   Duration                     820834 non-null  float64
 7   Scr_bytes                    820834 non-null  int64  
 8   Des_bytes                    820834 non-null  int64  
 9   Conn_state                   820834 non-null  int64  
 10  missed_bytes                 820834 non-null  float64
 11  is_syn_only                  820834 non-null  int64  
 12  Is_SYN_ACK                   820834 non-null  int64  
 13  is_pure_ac

In [2]:
df.head()

,Scr_IP,Scr_port,Des_IP,Des_port,Protocol,Service,Duration,Scr_bytes,Des_bytes,Conn_state,...,Login_attempt,Succesful_login,File_activity,Process_activity,read_write_physical.process,is_privileged,class1,class2,class3,Time
0,3.232236e+09,49278,3.232236e+09,80,1,4,0.673690,13437,34924,1,...,0,0,0,0,0,0,14,6,0,1578540956
1,1.677724e+08,39769,2.213283e+09,53,2,2,0.000083,78,0,1,...,0,0,0,0,0,0,11,4,1,1578871873
2,2.887254e+09,59050,2.887254e+09,53,2,2,0.000132,38,38,1,...,0,0,0,0,0,0,11,4,1,1578522486
3,3.232236e+09,37966,3.232236e+09,1880,1,16,9.378481,1121,484,1,...,1,1,1,1,1,1,11,4,1,1582757640
4,2.887254e+09,38233,2.887254e+09,53,2,2,0.000074,0,0,1,...,0,0,0,0,0,0,11,4,1,1576452612


In [3]:
df.describe()

,Scr_IP,Scr_port,Des_IP,Des_port,Protocol,Service,Duration,Scr_bytes,Des_bytes,Conn_state,...,Login_attempt,Succesful_login,File_activity,Process_activity,read_write_physical.process,is_privileged,class1,class2,class3,Time
count,8.208340e+05,820834.000000,8.208340e+05,820834.000000,820834.000000,820834.000000,820834.000000,8.208340e+05,8.208340e+05,820834.000000,...,820834.000000,820834.000000,820834.000000,820834.000000,820834.000000,820834.000000,820834.000000,820834.000000,820834.000000,8.208340e+05
mean,2.439797e+35,35635.117894,2.444278e+35,1746.322722,1.479027,4.524089,8.834584,1.500625e+03,4.104829e+04,0.851623,...,0.087305,0.082730,0.072650,0.082620,0.355309,0.082456,10.058964,4.696147,0.513401,1.229242e+09
std,9.081635e+36,21886.415529,9.098319e+36,4383.524522,0.506535,4.651246,113.660695,7.446342e+03,2.806029e+05,0.355474,...,0.282282,0.275475,0.259562,0.275307,0.478607,0.275059,3.622164,1.425801,0.499821,6.566496e+08
min,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00
25%,2.887254e+09,13307.000000,2.887254e+09,53.000000,1.000000,2.000000,0.000094,0.000000e+00,0.000000e+00,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,11.000000,4.000000,0.000000,1.576453e+09
50%,2.887255e+09,43011.000000,2.887254e+09,80.000000,1.000000,4.000000,0.004985,4.500000e+01,3.500000e+01,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,11.000000,4.000000,1.000000,1.578884e+09
75%,3.232236e+09,52593.000000,3.232236e+09,1880.000000,2.000000,4.000000,4.961307,6.080000e+02,3.100000e+02,1.000000,...,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,12.000000,5.000000,1.000000,1.581639e+09
max,3.382885e+38,65535.000000,3.389635e+38,65389.000000,3.000000,16.000000,9331.420034,1.665228e+06,1.935968e+07,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,18.000000,9.000000,1.000000,1.583570e+09


In [ ]:
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "svg"
target_cols = ['class1', 'class2', 'class3']

X = df.drop(columns=target_cols)
y = df[['class1']]  # Focus on class1 for now

# Create the 1,000 row set
X_train_small, X_test_small, y_train_small, y_test_small = train_test_split(
    X, y, train_size=800, test_size=200, stratify=y, random_state=42
)

X_train_med, X_test_med, y_train_med, y_test_med = train_test_split(
    X, y, train_size=8000, test_size=2000, stratify=y, random_state=42
)

# Create the 10,000 row set
X_train_large, X_test_large, y_train_large, y_test_large = train_test_split(
    X, y, train_size=80000, test_size=20000, stratify=y, random_state=42
)

labels = encoders['class1'].classes_
original_dist = df['class1'].value_counts(
    normalize=True).reindex(range(len(labels)), fill_value=0)
original_dist.index = labels
small_test_dist = y_test_small['class1'].value_counts(
    normalize=True).reindex(range(len(labels)), fill_value=0)
small_test_dist.index = labels
med_test_dist = y_test_med['class1'].value_counts(
    normalize=True).reindex(range(len(labels)), fill_value=0)
med_test_dist.index = labels
large_test_dist = y_test_large['class1'].value_counts(
    normalize=True).reindex(range(len(labels)), fill_value=0)
large_test_dist.index = labels

threshold = 0.01
common_classes = original_dist[original_dist > threshold].index
rare_classes = original_dist[original_dist <= threshold].index

# 2. Create subplots
fig = make_subplots(rows=2, cols=1,
                    subplot_titles=("Common Classes (>1%)",
                                    "Rare Classes (<1%)"),
                    vertical_spacing=0.15)

# Add traces for Common
for data, name, color in [(original_dist, 'Original', '#1f77b4'),
                          (small_test_dist, 'Small', '#ff7f0e'),
                          (med_test_dist, 'Medium', '#2ca02c'),
                          (large_test_dist, 'Large', '#d62728')]:
    fig.add_trace(go.Bar(x=common_classes,
                  y=data[common_classes], name=name, marker_color=color), row=1, col=1)

# Add traces for Rare
for data, name, color in [(original_dist, 'Original', '#1f77b4'),
                          (small_test_dist, 'Small', '#ff7f0e'),
                          (med_test_dist, 'Medium', '#2ca02c'),
                          (large_test_dist, 'Large', '#d62728')]:
    fig.add_trace(go.Bar(x=rare_classes, y=data[rare_classes],
                  name=name, marker_color=color, showlegend=False), row=2, col=1)

fig.update_layout(height=800, barmode='group',
                  title="Split Scale Distribution")

fig.write_image("Data_Distribution.svg")
fig.show()

pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)  # Ensure the width is large enough

orig_counts = df['class1'].value_counts()
orig_pcts = df['class1'].value_counts(normalize=True) * 100

# 2. Calculate counts and percentages for the Test Set
small_test_counts = y_test_small['class1'].value_counts()
small_test_pcts = y_test_small['class1'].value_counts(normalize=True) * 100
med_test_dist = y_test_med['class1'].value_counts()
med_test_pcts = y_test_med['class1'].value_counts(normalize=True) * 100
large_test_counts = y_test_large['class1'].value_counts()
large_test_pcts = y_test_large['class1'].value_counts(normalize=True) * 100

# 3. Combine into a single detailed DataFrame
comparison_table = pd.DataFrame({
    'Original (count)': orig_counts,
    'Original (%)': orig_pcts,
    'Small Test Set (count)': small_test_counts,
    'Small Test Set (%)': small_test_pcts,
    'Medium Test Set (count)': med_test_dist,
    'Medium Test Set (%)': med_test_pcts,
    'Large Test Set (count)': large_test_counts,
    'Large Test Set (%)': large_test_pcts
})


labels = encoders['class1'].classes_
comparison_table.index = [labels[i] for i in comparison_table.index]

# 5. Clean up the display
print("--- Detailed Distribution Table (class1) ---")
print(comparison_table.round(5))

NameError: name 'pio' is not defined

In [ ]:
from tabgan.sampler import GANGenerator, ForestDiffusionGenerator, LLMGenerator

categorical_features = ['Protocol', 'Service']


small_GAN_train, small_GAN_target = GANGenerator(gen_params={
    'cat_cols': categorical_features, "deep_copy": True, "only_generated_data": True,
    "batch_size": 200, "patience": 25, "epochs": 500, "gen_x_times": 2}) \
    .generate_data_pipe(train_df=X_train_small, target=y_train_small, test_df=X_test_small)
med_GAN_train, med_GAN_target = GANGenerator(gen_params={
    'cat_cols': categorical_features, "deep_copy": True, "only_generated_data": True,
    "batch_size": 500, "patience": 25, "epochs": 500, "gen_x_times": 2}) \
    .generate_data_pipe(train_df=X_train_med, target=y_train_med, test_df=X_test_med)
large_GAN_train, large_GAN_target = GANGenerator(gen_params={
    'cat_cols': categorical_features, "deep_copy": True, "only_generated_data": True,
    "batch_size": 500, "patience": 25, "epochs": 500, "gen_x_times": 2}) \
    .generate_data_pipe(train_df=X_train_large, target=y_train_large, test_df=X_test_large)

Training CTGAN, epochs::  12%|█▏        | 60/500 [15:25<1:53:05, 15.42s/it]


KeyboardInterrupt: 

In [ ]:
print(len(small_GAN_train), len(small_GAN_target),
      len(large_GAN_train), len(large_GAN_target))


labels = encoders['class1'].classes_

# 2. Use those labels as the index for your value counts
small_test_dist = pd.Series(small_GAN_target).value_counts(
    normalize=True).reindex(range(len(labels)), fill_value=0)
small_test_dist.index = labels
med_test_dist = pd.Series(med_GAN_target).value_counts(
    normalize=True).reindex(range(len(labels)), fill_value=0)
med_test_dist.index = labels
large_test_dist = pd.Series(large_GAN_target).value_counts(
    normalize=True).reindex(range(len(labels)), fill_value=0)
large_test_dist.index = labels

# 2. Create the Figure

fig = make_subplots(rows=2, cols=1,
                    subplot_titles=("Common Classes (>1%)",
                                    "Rare Classes (<1%)"),
                    vertical_spacing=0.15)

# Add traces for Common
for data, name, color in [(original_dist, 'Original', '#1f77b4'),
                          (small_test_dist, 'Small', '#ff7f0e'),
                          (med_test_dist, 'Medium', '#2ca02c'),
                          (large_test_dist, 'Large', '#d62728')]:
    fig.add_trace(go.Bar(x=common_classes,
                  y=data[common_classes], name=name, marker_color=color), row=1, col=1)

# Add traces for Rare
for data, name, color in [(original_dist, 'Original', '#1f77b4'),
                          (small_test_dist, 'Small', '#ff7f0e'),
                          (med_test_dist, 'Medium', '#2ca02c'),
                          (large_test_dist, 'Large', '#d62728')]:
    fig.add_trace(go.Bar(x=rare_classes, y=data[rare_classes],
                  name=name, marker_color=color), row=2, col=1)

fig.update_layout(height=800, barmode='group', xaxis={'type': 'category'},
                  title="Split Scale Distribution", showlegend=True, template='plotly_white')

fig.write_image("GAN_Generated_Distribution.svg")
fig.show()

small_test_counts = pd.Series(small_GAN_target).value_counts()
small_test_pcts = pd.Series(
    small_GAN_target).value_counts(normalize=True) * 100
med_test_dist = pd.Series(med_GAN_target).value_counts()
med_test_pcts = pd.Series(
    med_GAN_target).value_counts(normalize=True) * 100
large_test_counts = pd.Series(large_GAN_target).value_counts()
large_test_pcts = pd.Series(
    large_GAN_target).value_counts(normalize=True) * 100

# 3. Combine into a single detailed DataFrame
comparison_table = pd.DataFrame({
    'Original (count)': orig_counts,
    'Original (%)': orig_pcts,
    'Small Test Set (count)': small_test_counts,
    'Small Test Set (%)': small_test_pcts,
    'Med Test Set (count)': med_test_dist,
    'Med Test Set (%)': med_test_pcts,
    'Large Test Set (count)': large_test_counts,
    'Large Test Set (%)': large_test_pcts
})


labels = encoders['class1'].classes_
comparison_table.index = [labels[i] for i in comparison_table.index]

# 5. Clean up the display
print("--- Detailed Distribution Table (class1) ---")
print(comparison_table.round(5))

Fitting CTGAN transformers for each column:   0%|          | 0/65 [00:00<?, ?it/s]

Training CTGAN, epochs:: 100%|██████████| 500/500 [14:42<00:00,  1.76s/it]


624 624 7711 7711


--- Detailed Distribution Table (class1) ---
                                Original (count)  Original (%)  Small Test Set (count)  Small Test Set (%)  Large Test Set (count)  Large Test Set (%)
BruteForce                                 47241       5.75524                    41.0             6.57051                   446.0             5.78395
C&C                                         2863       0.34879                     NaN                 NaN                    25.0             0.32421
Dictionary                                  2572       0.31334                     2.0             0.32051                    25.0             0.32421
Discovering_resources                      23148       2.82006                    20.0             3.20513                   223.0             2.89197
Exfiltration                               22134       2.69653                    21.0             3.36538                   214.0             2.77526
Fake_notification                             28 